# Automated Segmentation of Vertebrae and Intervertebral Discs in Lumbar Spine MRI Images

## Sprint 1 - Dataset Inspection and Preprocessing

**Problem.** Manual segmentation of lumbar spine MRI is slow and varies between
observers, which limits both clinical throughput and the reproducibility of any
measurement derived from it.

**Objective.** Build an automated pipeline that segments vertebrae and
intervertebral discs in sagittal lumbar spine MRI.

**Scope of this notebook.** Sprint 1 only: inspect the dataset, extract it, pair
images with masks, validate quality, preprocess images and masks, visualise the
result, and create a leakage-free train/validation/test split.

> **No segmentation model is trained here, and no Dice or IoU score is reported.**
> Those belong to the next sprint. Any accuracy figure would be meaningless until
> a model has actually been trained and evaluated.

### Pipeline overview

```
data/raw/*.zip
      |  extract.py
      v
data/extracted/{images,masks}/*.mha      447 sagittal 3-D volumes
      |  validate.py        -> dataset_inspection.md
      |  pairing.py         -> image_mask_pairs.csv
      v
transforms.py + dataset.py               reorient -> slice -> resample -> crop
      |                                  -> normalise -> denoise -> CLAHE
      v
data/processed/slices/*.npz              fixed-size 2-D image/mask pairs
      |  splits.py
      v
data/processed/splits.csv                patient-level train/val/test
```

### How this notebook is organised

1. Project introduction
2. Dataset description
3. Dataset inspection
4. Data extraction
5. Image-mask pairing
6. Dataset quality checks
7. Image preprocessing
8. Mask preprocessing
9. Before/after visualisation
10. Train/validation/test split
11. Final preprocessing summary

### Setup

The only thing that needs configuring is the project root. Everything else is
derived from it by `src/utils/paths.py`, so the notebook runs start-to-finish
without further manual intervention.

In [1]:
%matplotlib inline

import sys
from pathlib import Path

# Make the project importable whether the notebook is run from notebooks/ or
# from the project root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

from src.utils import paths

print("Project root:", paths.PROJECT_ROOT)
print("Raw data    :", paths.RAW_DIR)
print("Random seed :", paths.RANDOM_SEED)

Project root: D:\healthcare analytics\lumbar-spine-segmentation
Raw data    : D:\healthcare analytics\lumbar-spine-segmentation\data\raw
Random seed : 42


---
## 2. Dataset description

The project dataset is a **lumbar spine MRI segmentation dataset** supplied as
four files in `data/raw/`:

| file | content |
| --- | --- |
| `images.zip` | MRI volumes |
| `masks.zip` | matching segmentation masks |
| `overview.csv` | one row per series: acquisition metadata and structure counts |
| `radiological_gradings.csv` | one row per patient x disc: radiological gradings |

This is the publicly described lumbar spine MRI segmentation dataset of sagittal
T1 and T2 series ([SPIDER, Grand Challenge](https://spider.grand-challenge.org/data/);
[van der Graaf et al., *Scientific Data* 11, 264, 2024](https://www.nature.com/articles/s41597-024-03090-w)).
Everything stated below about format, labels and geometry was nevertheless
**measured from the files themselves** rather than taken from the description.

> The two notebooks in `tutorials/` use a completely different Electronic Health
> Record dataset. They are kept as reference material only and are not used here.

In [2]:
for path in [paths.IMAGES_ZIP, paths.MASKS_ZIP, paths.OVERVIEW_CSV, paths.GRADINGS_CSV]:
    size = path.stat().st_size / 1e6 if path.exists() else float("nan")
    print(f"{path.name:28s} exists={path.exists()}  {size:>9,.2f} MB")

images.zip                   exists=True   3,700.56 MB
masks.zip                    exists=True      58.22 MB
overview.csv                 exists=True       0.12 MB
radiological_gradings.csv    exists=True       0.03 MB


### 2.1 Metadata files

`overview.csv` is keyed on `new_file_name`, which matches the volume filename
stem exactly, so it joins directly onto the image files.
`radiological_gradings.csv` is keyed on `Patient`, the numeric prefix of the
filename.

The loaders in `src/preprocessing/pairing.py` also clean one real defect found
during inspection: `sex` is stored with inconsistent trailing whitespace, so
`'F'` and `'F '` appear as two distinct categories.

In [3]:
from src.preprocessing.pairing import load_gradings, load_overview

raw_overview = pd.read_csv(paths.OVERVIEW_CSV)
overview = load_overview()   # whitespace-stripped + patient_id / modality parsed
gradings = load_gradings()

print(f"overview.csv : {raw_overview.shape[0]} rows x {raw_overview.shape[1]} columns")
print(f"gradings.csv : {gradings.shape[0]} rows, {gradings['patient_id'].nunique()} patients")
print("\nsex BEFORE cleaning:", raw_overview["sex"].value_counts(dropna=False).to_dict())
print("sex AFTER cleaning :", overview["sex"].value_counts(dropna=False).to_dict())

overview[["new_file_name", "patient_id", "modality", "num_vertebrae", "num_discs",
          "sex", "subset", "Manufacturer", "PixelSpacing", "SliceThickness"]].head()

overview.csv : 447 rows x 39 columns
gradings.csv : 1520 rows, 218 patients

sex BEFORE cleaning: {'F': 152, 'F ': 124, 'M': 115, 'M ': 56}
sex AFTER cleaning : {'F': 276, 'M': 171}


,new_file_name,patient_id,modality,num_vertebrae,num_discs,sex,subset,Manufacturer,PixelSpacing,SliceThickness
0,1_t1,1,t1,7,7,F,training,Philips Healthcare,"[0.625, 0.625]",3.0
1,1_t2,1,t2,7,7,F,training,Philips Healthcare,"[0.625, 0.625]",3.0
2,10_t1,10,t1,7,7,F,training,SIEMENS,"[0.3125, 0.3125]",4.0
3,10_t2,10,t2,7,7,F,training,SIEMENS,"[0.36458334326744, 0.36458334326744]",4.0
4,100_t1,100,t1,8,8,F,training,Philips Medical Systems,"[0.47826087474823, 0.47826087474823]",4.0


In [4]:
# Per-disc radiological gradings: classification targets for a possible later
# sprint, not segmentation targets. Summarised here only for completeness.
print("Gradings columns:", list(gradings.columns))
display(gradings.head())
display(gradings["pfirrman_grade"].value_counts().sort_index().rename("n discs").to_frame())

Gradings columns: ['patient_id', 'ivd_label', 'modic', 'up_endplate', 'low_endplate', 'spondylolisthesis', 'disc_herniation', 'disc_narrowing', 'disc_bulging', 'pfirrman_grade']


,patient_id,ivd_label,modic,up_endplate,low_endplate,spondylolisthesis,disc_herniation,disc_narrowing,disc_bulging,pfirrman_grade
0,1,1,0,0,0,0,0,1,1,3
1,1,2,0,0,0,0,0,0,1,3
2,1,3,0,0,0,0,0,1,1,3
3,1,4,0,0,0,0,0,1,1,4
4,1,5,0,0,0,0,0,1,0,4


,n discs
pfirrman_grade,
1,286
2,341
3,418
4,291
5,184


---
## 3. Dataset inspection

Nothing about the dataset was assumed. Before writing any preprocessing code the
archives were opened and examined to determine the file format, the array
geometry, the label vocabulary and the intensity conventions.

### 3.1 What is inside the archives

In [5]:
import os
from collections import Counter

from src.preprocessing.extract import list_archive

for zip_path in [paths.IMAGES_ZIP, paths.MASKS_ZIP]:
    names = [n for n in list_archive(zip_path) if not n.endswith("/")]
    extensions = Counter(os.path.splitext(n)[1].lower() for n in names)
    print(f"{zip_path.name}: {len(names)} members, extensions={dict(extensions)}")
    print("   first 3:", names[:3])

images.zip: 447 members, extensions={'.mha': 447}
   first 3: ['images/1_t1.mha', 'images/1_t2.mha', 'images/10_t1.mha']
masks.zip: 447 members, extensions={'.mha': 447}
   first 3: ['masks/1_t1.mha', 'masks/1_t2.mha', 'masks/10_t1.mha']


**Finding.** Both archives contain **447 `.mha` files** - MetaImage format, which
holds a *3-D volume* plus its physical geometry. So this is not a folder of 2-D
PNG/JPEG images: each file is a sagittal MRI series, and the project's 2-D
segmentation targets have to be produced by slicing the volumes.

Filenames follow `<patient_id>_<modality>.mha`, and `images/X.mha` corresponds to
`masks/X.mha`.

In [6]:
from src.preprocessing.pairing import parse_series_stem

stems = [Path(n).stem for n in list_archive(paths.IMAGES_ZIP) if n.endswith(".mha")]
parsed = [parse_series_stem(s) for s in stems]
patient_ids = sorted({p for p, _ in parsed})
modalities = Counter(m for _, m in parsed)
series_per_patient = Counter(Counter(p for p, _ in parsed).values())

print(f"Series            : {len(stems)}")
print(f"Distinct patients : {len(patient_ids)}  (ids {min(patient_ids)}..{max(patient_ids)}, not contiguous)")
print(f"Modalities        : {dict(modalities)}")
print(f"Series per patient: {dict(sorted(series_per_patient.items()))}")

Series            : 447
Distinct patients : 218  (ids 1..257, not contiguous)
Modalities        : {'t1': 196, 't2': 210, 't2_SPACE': 41}
Series per patient: {1: 28, 2: 151, 3: 39}


**Finding that shapes the whole project.** There are **447 series but only 218
patients** - most patients contribute 2 or 3 series of the *same* anatomy. This
is why the train/validation/test split later has to be made per patient rather
than per series or per slice.

---
## 4. Data extraction

Extraction is done by code (`src/preprocessing/extract.py`) so it is
reproducible and the raw ZIPs are never modified. The function is resume-safe:
a member whose target file already exists at the expected size is skipped, so
re-running is cheap.

In [7]:
from src.preprocessing.extract import extract_dataset

# Resume-safe: skips files that already exist with the expected size.
counts = extract_dataset(verbose=False)
print("Extracted:", counts)
print("images ->", paths.EXTRACTED_IMAGES_DIR)
print("masks  ->", paths.EXTRACTED_MASKS_DIR)

# The raw archives are untouched.
print("\nRaw archives still present:",
      paths.IMAGES_ZIP.exists(), paths.MASKS_ZIP.exists())

Extracted: {'images': 447, 'masks': 447}
images -> D:\healthcare analytics\lumbar-spine-segmentation\data\extracted\images
masks  -> D:\healthcare analytics\lumbar-spine-segmentation\data\extracted\masks

Raw archives still present: True True


### 4.1 Reading a volume: geometry is not consistent

Loading is handled by `src/preprocessing/volume_io.py`. The important detail it
solves: the volumes are **not all stored with the same axis order**. Most 2-D
TSE series are stored `LPS`, while the 3-D `t2_SPACE` series are stored `PIR`,
so the array axis that steps through sagittal slices differs between files.

Every volume is therefore reoriented to a canonical `RAS` frame on load. That is
a pure axis permutation plus flips - no interpolation - so it is exactly
loss-less and safe to apply to label masks.

In [8]:
from src.preprocessing.volume_io import load_image, load_mask, volume_geometry

for stem in ["1_t1", "98_t2_SPACE"]:
    image = load_image(paths.EXTRACTED_IMAGES_DIR / f"{stem}.mha")
    mask = load_mask(paths.EXTRACTED_MASKS_DIR / f"{stem}.mha")
    print(f"--- {stem}")
    print(f"    stored orientation : {image.native_orientation}")
    print(f"    array shape (z,y,x): {image.array.shape}   dtype={image.array.dtype}")
    print(f"    spacing mm  (z,y,x): {tuple(round(s, 3) for s in image.spacing_zyx)}")
    print(f"    sagittal slices    : {image.n_sagittal_slices}")
    print(f"    slice shape        : {image.in_plane_shape}")
    print(f"    mask shape matches : {image.array.shape == mask.array.shape}")
    print(f"    raw intensity range: [{image.array.min():.0f}, {image.array.max():.0f}]")
    print(f"    mask label values  : {np.unique(mask.array).tolist()}")

--- 1_t1
    stored orientation : LPS
    array shape (z,y,x): (578, 448, 50)   dtype=float32
    spacing mm  (z,y,x): (0.501, 0.625, 3.321)
    sagittal slices    : 50
    slice shape        : (578, 448)
    mask shape matches : True
    raw intensity range: [-1000, 3096]
    mask label values  : [0, 1, 2, 3, 4, 5, 6, 7, 100, 201, 202, 203, 204, 205, 206, 207]
--- 98_t2_SPACE
    stored orientation : PIR
    array shape (z,y,x): (640, 512, 120)   dtype=float32
    spacing mm  (z,y,x): (0.469, 0.469, 0.9)
    sagittal slices    : 120
    slice shape        : (640, 512)
    mask shape matches : True
    raw intensity range: [0, 610]
    mask label values  : [0, 1, 2, 3, 4, 5, 6, 7, 100, 201, 202, 203, 204, 205, 206, 207, 208]


**Two findings visible above.**

1. **Label vocabulary is sparse and non-contiguous**: `0`, `1..9`, `100`,
   `201..209`. Masks are integer label maps, not binary and not one-hot.
2. **Intensity conventions differ**: one series runs `[-1000, 3096]`, the other
   `[0, ...]`. MRI intensity has no absolute meaning, so normalisation is
   mandatory rather than optional.

### 4.2 Label semantics

Documented and implemented in `src/preprocessing/labels.py`:

| raw value | meaning |
| --- | --- |
| `0` | background |
| `1..9` | vertebrae, numbered from the most inferior upward |
| `100` | spinal canal |
| `201..209` | intervertebral discs, numbered from the most inferior upward |

Two derived label spaces are produced:

* **semantic** (the project target): `0` background, `1` vertebra, `2` IVD, `3` spinal canal
* **instance** (loss-less): `0` background, `1..9` vertebrae, `10` canal, `11..19` IVDs

In [9]:
from src.preprocessing.labels import (
    SEMANTIC_CLASSES, describe_raw_label, to_instance, to_semantic,
)

mask = load_mask(paths.EXTRACTED_MASKS_DIR / "1_t1.mha")
for value in np.unique(mask.array):
    print(f"  {int(value):>4} -> {describe_raw_label(int(value))}")

print("\nSemantic classes:", SEMANTIC_CLASSES)
print("Raw label values      :", np.unique(mask.array).tolist())
print("Semantic label values :", np.unique(to_semantic(mask.array)).tolist())
print("Instance label values :", np.unique(to_instance(mask.array)).tolist())

     0 -> background
     1 -> vertebra #1 (counted from the most inferior)
     2 -> vertebra #2 (counted from the most inferior)
     3 -> vertebra #3 (counted from the most inferior)
     4 -> vertebra #4 (counted from the most inferior)
     5 -> vertebra #5 (counted from the most inferior)
     6 -> vertebra #6 (counted from the most inferior)
     7 -> vertebra #7 (counted from the most inferior)
   100 -> spinal canal
   201 -> IVD #1 (counted from the most inferior)
   202 -> IVD #2 (counted from the most inferior)
   203 -> IVD #3 (counted from the most inferior)
   204 -> IVD #4 (counted from the most inferior)
   205 -> IVD #5 (counted from the most inferior)
   206 -> IVD #6 (counted from the most inferior)
   207 -> IVD #7 (counted from the most inferior)

Semantic classes: {0: 'background', 1: 'vertebra', 2: 'intervertebral_disc', 3: 'spinal_canal'}
Raw label values      : [0, 1, 2, 3, 4, 5, 6, 7, 100, 201, 202, 203, 204, 205, 206, 207]
Semantic label values : [0, 1, 2, 3

---
## 5. Image-mask pairing

Pairing is driven by the **parsed filename identifier**, never by directory
listing order. Alphabetical pairing would be dangerous here: a single missing
file would silently shift every subsequent pair by one and every later metric
would be quietly wrong.

In [10]:
from src.preprocessing.pairing import describe_pairing, pair_images_and_masks

pairs, pairing_issues = pair_images_and_masks(overview=overview, gradings=gradings)
print(describe_pairing(pairs, pairing_issues, n_examples=5))

Matched image/mask pairs : 447
Distinct patients        : 218
Modalities               : {'t2': 210, 't1': 196, 't2_SPACE': 41}
Images without a mask    : 0
Masks without an image   : 0
Unparsable filenames     : 0
Missing overview.csv row : 0

Example pairs (first 5):
  [1_t1] patient=1 modality=t1
      image: data/extracted/images/1_t1.mha
      mask : data/extracted/masks/1_t1.mha
  [1_t2] patient=1 modality=t2
      image: data/extracted/images/1_t2.mha
      mask : data/extracted/masks/1_t2.mha
  [2_t1] patient=2 modality=t1
      image: data/extracted/images/2_t1.mha
      mask : data/extracted/masks/2_t1.mha
  [2_t2] patient=2 modality=t2
      image: data/extracted/images/2_t2.mha
      mask : data/extracted/masks/2_t2.mha
  [3_t1] patient=3 modality=t1
      image: data/extracted/images/3_t1.mha
      mask : data/extracted/masks/3_t1.mha


In [11]:
# The pairing table: what later sprints consume.
print("Columns:", list(pairs.columns))
pairs.head(8)

Columns: ['image_id', 'patient_id', 'modality', 'image_path', 'mask_path', 'num_vertebrae', 'num_discs', 'sex', 'subset', 'Manufacturer', 'ManufacturerModelName', 'MagneticFieldStrength', 'MRAcquisitionType', 'ScanningSequence', 'SeriesDescription', 'PixelSpacing', 'SliceThickness', 'SpacingBetweenSlices', 'EchoTime', 'RepetitionTime', 'n_graded_discs', 'max_pfirrman_grade']


,image_id,patient_id,modality,image_path,mask_path,num_vertebrae,num_discs,sex,subset,Manufacturer,ManufacturerModelName,MagneticFieldStrength,MRAcquisitionType,ScanningSequence,SeriesDescription,PixelSpacing,SliceThickness,SpacingBetweenSlices,EchoTime,RepetitionTime,n_graded_discs,max_pfirrman_grade
0,1_t1,1,t1,data/extracted/images/1_t1.mha,data/extracted/masks/1_t1.mha,7,7,F,training,Philips Healthcare,Ingenia,3.0,2D,SE,NaN,"[0.625, 0.625]",3.0,3.3,13.837,716.234375,7,4
1,1_t2,1,t2,data/extracted/images/1_t2.mha,data/extracted/masks/1_t2.mha,7,7,F,training,Philips Healthcare,Ingenia,3.0,2D,SE,NaN,"[0.625, 0.625]",3.0,3.3,120.000,3563.072510,7,4
2,2_t1,2,t1,data/extracted/images/2_t1.mha,data/extracted/masks/2_t1.mha,6,6,M,validation,SIEMENS,Avanto_fit,1.5,2D,SE,t1_tse_sag_320,"[0.8125, 0.8125]",4.0,4.8,9.600,660.000000,6,4
3,2_t2,2,t2,data/extracted/images/2_t2.mha,data/extracted/masks/2_t2.mha,6,6,M,validation,SIEMENS,Avanto_fit,1.5,2D,SE,t2_tse_sag_384,"[0.67708331346512, 0.67708331346512]",4.0,4.8,114.000,3500.000000,6,4
4,3_t1,3,t1,data/extracted/images/3_t1.mha,data/extracted/masks/3_t1.mha,6,6,F,training,SIEMENS,Avanto_fit,1.5,2D,SE,t1_tse_sag_320,"[0.8125, 0.8125]",4.0,4.8,9.600,687.000000,6,4
5,3_t2,3,t2,data/extracted/images/3_t2.mha,data/extracted/masks/3_t2.mha,6,6,F,training,SIEMENS,Avanto_fit,1.5,2D,SE,t2_tse_sag_384,"[0.67708331346512, 0.67708331346512]",4.0,4.8,114.000,3680.000000,6,4
6,4_t1,4,t1,data/extracted/images/4_t1.mha,data/extracted/masks/4_t1.mha,6,6,M,training,Philips Healthcare,Ingenia,3.0,2D,SE,NaN,"[0.625, 0.625]",3.0,3.3,13.837,716.234375,6,5
7,4_t2,4,t2,data/extracted/images/4_t2.mha,data/extracted/masks/4_t2.mha,6,6,M,training,Philips Healthcare,Ingenia,3.0,2D,SE,NaN,"[0.62499964237213, 0.62499964237213]",3.0,3.3,120.000,3563.072510,6,5


In [12]:
# Verify every pair explicitly: the mask file must exist and describe the same grid.
import random

random.seed(paths.RANDOM_SEED)
checks = []
for _ in range(5):
    row = pairs.iloc[random.randrange(len(pairs))]
    image = load_image(paths.PROJECT_ROOT / row["image_path"])
    mask = load_mask(paths.PROJECT_ROOT / row["mask_path"])
    checks.append({
        "image_id": row["image_id"],
        "patient_id": row["patient_id"],
        "modality": row["modality"],
        "same_stem": Path(row["image_path"]).stem == Path(row["mask_path"]).stem,
        "same_shape": image.array.shape == mask.array.shape,
        "same_spacing": np.allclose(image.spacing_zyx, mask.spacing_zyx, atol=1e-4),
        "n_labels": int(len(np.unique(mask.array))),
    })
pd.DataFrame(checks)

,image_id,patient_id,modality,same_stem,same_shape,same_spacing,n_labels
0,188_t1,188,t1,True,True,True,16
1,31_t2,31,t2,True,True,True,18
2,7_t1,7,t1,True,True,True,16
3,219_t1,219,t1,True,True,True,16
4,74_t2,74,t2,True,True,True,14


---
## 6. Dataset quality checks

`scripts/02_inspect.py` reads every one of the 447 volume pairs and records
geometry, intensity, labels and integrity into
`outputs/preprocessing_reports/`. That scan takes several minutes, so this
notebook loads its results rather than repeating it.

Run it first if the files are missing:

```bash
python scripts/02_inspect.py
```

In [13]:
import json

inspection = pd.read_csv(paths.VOLUME_INSPECTION_CSV)
with open(paths.REPORTS_DIR / "dataset_inspection.json") as handle:
    inspection_summary = json.load(handle)

print(f"Inspected series : {len(inspection)}")
print(f"Read errors      : {int(inspection['read_error'].notna().sum())}")
print(f"Distinct patients: {inspection_summary['n_patients']}")
print(f"Label vocabulary : {inspection_summary['label_vocabulary']}")
print(f"Undocumented labels: {inspection_summary['unknown_labels'] or 'none'}")

Inspected series : 447
Read errors      : 0
Distinct patients: 218
Label vocabulary : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 100, 201, 202, 203, 204, 205, 206, 207, 208, 209]
Undocumented labels: none


### 6.1 Missing, unmatched and duplicate files

In [14]:
duplicates = inspection_summary["duplicates"]
quality = {
    "images without a mask": len(inspection_summary["pairing_issues"]["images_without_mask"]),
    "masks without an image": len(inspection_summary["pairing_issues"]["masks_without_image"]),
    "filenames not matching convention": len(inspection_summary["pairing_issues"]["unparsable_stems"]),
    "series missing an overview.csv row": len(inspection_summary["pairing_issues"]["missing_from_overview"]),
    "duplicate IMAGE groups": duplicates["images"]["n_duplicate_groups"],
    "duplicate MASK groups": duplicates["masks"]["n_duplicate_groups"],
    "duplicate mask groups within one patient": duplicates["masks"]["n_same_patient_groups"],
    "duplicate mask groups across patients": duplicates["masks"]["n_cross_patient_groups"],
    "image/mask shape disagreements": inspection_summary["geometry_mismatches"]["shape"],
    "image/mask spacing disagreements": inspection_summary["geometry_mismatches"]["spacing"],
}
pd.Series(quality, name="count").to_frame()

,count
images without a mask,0
masks without an image,0
filenames not matching convention,0
series missing an overview.csv row,0
duplicate IMAGE groups,0
duplicate MASK groups,107
duplicate mask groups within one patient,107
duplicate mask groups across patients,0
image/mask shape disagreements,0
image/mask spacing disagreements,0


**Interpretation of the duplicates.** No *image* volume is duplicated. Many
*mask* volumes are byte-identical, but **always within a single patient** - the
T1 and T2 series of a patient were acquired on the same grid and share one
annotation. That is a property of the annotation process, not corruption. It
does mean a patient's series are highly correlated, which is a second
independent reason the split must be per patient.

### 6.2 Anatomical sanity checks

These verify that the reorientation actually put the axes where the code assumes
they are. They are derived from mask geometry alone:

* the spinal canal must lie **posterior** to the vertebral bodies
* vertebra label `1` must lie **inferior** to the highest vertebra label

In [15]:
checks = inspection_summary["anatomical_checks"]
pd.DataFrame([
    {"check": "spinal canal posterior to vertebral bodies",
     "passed": checks["canal_posterior_pass"], "failed": checks["canal_posterior_fail"]},
    {"check": "vertebra label 1 inferior to highest vertebra label",
     "passed": checks["label1_inferior_pass"], "failed": checks["label1_inferior_fail"]},
])

,check,passed,failed
0,spinal canal posterior to vertebral bodies,447,0
1,vertebra label 1 inferior to highest vertebra ...,447,0


### 6.3 Heterogeneity: why a naive resize would be wrong

In [16]:
ok = inspection[inspection["read_error"].isna()]

summary = pd.DataFrame({
    "min": [ok["img_rows"].min(), ok["img_cols"].min(), ok["img_n_slices"].min(),
            ok["img_row_spacing_mm"].min(), ok["img_col_spacing_mm"].min(),
            ok["img_slice_spacing_mm"].min()],
    "median": [ok["img_rows"].median(), ok["img_cols"].median(), ok["img_n_slices"].median(),
               ok["img_row_spacing_mm"].median(), ok["img_col_spacing_mm"].median(),
               ok["img_slice_spacing_mm"].median()],
    "max": [ok["img_rows"].max(), ok["img_cols"].max(), ok["img_n_slices"].max(),
            ok["img_row_spacing_mm"].max(), ok["img_col_spacing_mm"].max(),
            ok["img_slice_spacing_mm"].max()],
}, index=["rows (px)", "cols (px)", "slices", "row spacing (mm)",
          "col spacing (mm)", "slice spacing (mm)"])

print(f"Distinct in-plane shapes: {ok.groupby(['img_rows', 'img_cols']).ngroups}")
print(f"Stored orientations     : {inspection_summary['native_orientations']}")
summary.round(4)

Distinct in-plane shapes: 149
Stored orientations     : {'LPS': 374, 'PIR': 73}


,min,median,max
rows (px),216.0000,478.0000,3682.0000
cols (px),264.0000,448.0000,1168.0000
slices,8.0000,24.0000,154.0000
row spacing (mm),0.0767,0.5874,1.2331
col spacing (mm),0.2480,0.6250,1.0625
slice spacing (mm),0.8586,3.3097,9.6273


In [17]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist(ok["img_rows"], bins=40, color="#4c72b0")
axes[0].set_title("In-plane rows (px)"); axes[0].set_xlabel("pixels")
axes[1].hist(ok["img_row_spacing_mm"], bins=40, color="#dd8452")
axes[1].set_title("In-plane pixel spacing (mm)"); axes[1].set_xlabel("mm")
axes[2].hist(ok["img_n_slices"], bins=40, color="#55a868")
axes[2].set_title("Sagittal slices per volume"); axes[2].set_xlabel("slices")
for axis in axes:
    axis.set_ylabel("series")
fig.suptitle("Acquisition heterogeneity across the 447 series", y=1.04)
fig.tight_layout()
plt.show()

<Figure size 1500x380 with 3 Axes>

Pixel spacing spans more than an order of magnitude. Resizing purely by pixel
count would therefore place the same vertebra at a different physical size in
different patients. Preprocessing resamples to a **fixed mm/pixel** instead.

### 6.4 The two intensity conventions

In [18]:
convention = ok.groupby(["raw_min", "raw_max"]).size().rename("n series").reset_index()
display(convention)

print(f"Series with a constant padding floor: {int(ok['padding_value'].notna().sum())} / {len(ok)}")
print(f"Mean foreground fraction            : {ok['foreground_fraction'].mean():.1%}")
print(f"Mean fraction pinned at the maximum : {ok['saturated_fraction'].mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for axis, stem in zip(axes, ["1_t1", "98_t2_SPACE"]):
    volume = load_image(paths.EXTRACTED_IMAGES_DIR / f"{stem}.mha")
    axis.hist(volume.array.ravel(), bins=150, color="#666666")
    axis.set_yscale("log")
    axis.set_title(f"{stem}: raw intensity histogram\n"
                   f"range [{volume.array.min():.0f}, {volume.array.max():.0f}]")
    axis.set_xlabel("raw intensity"); axis.set_ylabel("voxels (log)")
fig.tight_layout()
plt.show()

,raw_min,raw_max,n series
0,-1000.0,3096.0,374
1,0.0,349.0,1
2,0.0,379.0,1
3,0.0,398.0,1
4,0.0,408.0,1
...,...,...,...
65,0.0,922.0,1
66,0.0,962.0,1
67,0.0,1345.0,1
68,0.0,1367.0,1


Series with a constant padding floor: 447 / 447
Mean foreground fraction            : 74.4%
Mean fraction pinned at the maximum : 4.20%


<Figure size 1200x380 with 2 Axes>

The left histogram shows the problem: a huge spike of background voxels at the
padding floor and another at the saturated ceiling. A mean/std or min-max
normalisation would be governed by those two spikes rather than by tissue, which
is why statistics are computed over **foreground voxels only**.

### 6.5 Class imbalance

Recorded now because it will decide the loss function in the modelling sprint.

In [19]:
totals = {
    "vertebrae": float(ok["voxels_vertebrae"].sum()),
    "intervertebral discs": float(ok["voxels_ivd"].sum()),
    "spinal canal": float(ok["voxels_canal"].sum()),
}
labelled = sum(totals.values())
imbalance = pd.DataFrame({
    "voxels": {k: int(v) for k, v in totals.items()},
    "% of labelled voxels": {k: round(100 * v / labelled, 2) for k, v in totals.items()},
})
print(f"Labelled voxels as a share of all voxels: "
      f"{ok['labelled_voxel_fraction'].mean():.2%} (mean per volume)")
imbalance

Labelled voxels as a share of all voxels: 5.64% (mean per volume)


,voxels,% of labelled voxels
vertebrae,148114894,71.70
intervertebral discs,31440866,15.22
spinal canal,27022473,13.08


---
## 7. Image preprocessing

Every step below was chosen from measurements, not from a generic recipe. The
supporting evidence is in
`outputs/preprocessing_reports/preprocessing_choices.md`, produced by
`scripts/02b_justify_steps.py`, which scores seven competing variants on real
slices.

| step | applied? | why |
| --- | --- | --- |
| reorient to RAS | **yes** | stored orientation varies (`LPS` / `PIR`); loss-less |
| resample to 1.0 mm/px | **yes** | pixel spacing varies 0.077-1.23 mm |
| centre crop/pad to 352x256 | **yes** | measured: contains all annotation in all 447 volumes |
| percentile normalisation | **yes** | two incompatible intensity conventions |
| median 3x3 denoise | **yes** | best class contrast; keeps ~80% of edge sharpness |
| CLAHE | **yes** | highest vertebra-vs-disc contrast of all variants tested |
| N4 bias field correction | **no** | measured effect negligible; CLAHE already halves inhomogeneity |

### 7.1 Why the target size is 352 x 256 and not a square

For every volume, the smallest **centred** crop that still contains the entire
annotation was computed. The lumbar spine is tall and narrow, so the requirement
is very different along the two axes.

In [20]:
requirements = {}
for axis_name, frac_col, extent_col, fov_col in [
    ("superior-inferior (rows)", "ann_row_center_frac", "ann_rows_mm", "img_fov_rows_mm"),
    ("anterior-posterior (cols)", "ann_col_center_frac", "ann_cols_mm", "img_fov_cols_mm"),
]:
    fov = ok[fov_col].to_numpy()
    centre = ok[frac_col].to_numpy() * fov
    extent = ok[extent_col].to_numpy()
    # Smallest centred window that still contains [centre-extent/2, centre+extent/2].
    needed = 2 * np.maximum(np.abs(centre - extent / 2 - fov / 2),
                            np.abs(centre + extent / 2 - fov / 2))
    requirements[axis_name] = needed
    print(f"{axis_name}: annotation extent {extent.min():.0f}-{extent.max():.0f} mm, "
          f"required centred crop max {needed.max():.0f} mm")

print()
rows = []
for size in [256, 288, 320, 352, 384]:
    rows.append({
        "crop (mm)": size,
        "volumes losing annotation (rows)": int((requirements["superior-inferior (rows)"] > size).sum()),
        "volumes losing annotation (cols)": int((requirements["anterior-posterior (cols)"] > size).sum()),
    })
pd.DataFrame(rows)

superior-inferior (rows): annotation extent 107-291 mm, required centred crop max 340 mm
anterior-posterior (cols): annotation extent 77-133 mm, required centred crop max 244 mm



,crop (mm),volumes losing annotation (rows),volumes losing annotation (cols)
0,256,447,0
1,288,157,0
2,320,4,0
3,352,0,0
4,384,0,0


A square `288 x 288` crop would have clipped annotated anatomy in **157 of 447
volumes**. `352` rows x `256` cols loses nothing, and both values are multiples
of 32, which suits the downsampling depth of a U-Net later.

### 7.2 The configuration

All parameters live in one dataclass, so the pipeline is auditable and
reproducible.

In [21]:
from src.preprocessing.transforms import PreprocessConfig

config = PreprocessConfig()
pd.Series(config.to_dict(), name="value").to_frame()

,value
target_spacing_mm,1.0
target_size,"(352, 256)"
clip_percentiles,"(1.0, 99.0)"
normalize,True
denoise,True
denoise_method,median
denoise_kernel,3
enhance_contrast,True
clahe_clip_limit,2.0
clahe_tile_grid,"(8, 8)"


### 7.3 The pipeline stage by stage

Order matters: geometry first (so a 3x3 kernel means the same physical size
everywhere), then normalisation, then denoising, and CLAHE last because CLAHE
amplifies whatever noise is present.

In [22]:
from src.preprocessing.transforms import (
    apply_geometry, denoise_image, enhance_contrast, intensity_statistics, normalize_image,
)

row = pairs[pairs["modality"] == "t2"].iloc[0]
image_volume = load_image(paths.PROJECT_ROOT / row["image_path"])
mask_volume = load_mask(paths.PROJECT_ROOT / row["mask_path"])

# Use the slice with the most annotation - the informative mid-sagittal slice.
slice_index = int(np.argmax((mask_volume.array > 0).sum(axis=(0, 1))))
spacing = image_volume.in_plane_spacing
volume_stats = intensity_statistics(image_volume.array, percentiles=config.clip_percentiles)

raw_slice = image_volume.sagittal_slice(slice_index)
geometry = apply_geometry(raw_slice, spacing, config, is_mask=False)
normalised = normalize_image(geometry, percentiles=config.clip_percentiles, stats=volume_stats)
denoised = denoise_image(normalised, config.denoise_method, config.denoise_kernel)
final = enhance_contrast(denoised, clip_limit=config.clahe_clip_limit,
                         tile_grid=config.clahe_tile_grid)

print(f"series {row['image_id']}, sagittal slice {slice_index}")
pd.DataFrame([
    {"stage": "1. raw slice", "shape": raw_slice.shape,
     "min": raw_slice.min(), "max": raw_slice.max(), "mean": raw_slice.mean()},
    {"stage": f"2. resampled to {config.target_spacing_mm} mm + crop", "shape": geometry.shape,
     "min": geometry.min(), "max": geometry.max(), "mean": geometry.mean()},
    {"stage": "3. normalised", "shape": normalised.shape,
     "min": normalised.min(), "max": normalised.max(), "mean": normalised.mean()},
    {"stage": f"4. {config.denoise_method} denoise", "shape": denoised.shape,
     "min": denoised.min(), "max": denoised.max(), "mean": denoised.mean()},
    {"stage": "5. CLAHE (final)", "shape": final.shape,
     "min": final.min(), "max": final.max(), "mean": final.mean()},
]).round(4)

series 1_t2, sagittal slice 22


,stage,shape,min,max,mean
0,1. raw slice,"(578, 448)",-1000.0000,3096.0,237.460098
1,2. resampled to 1.0 mm + crop,"(352, 256)",-1000.0000,3096.0,290.208496
2,3. normalised,"(352, 256)",0.0000,1.0,0.313500
3,4. median denoise,"(352, 256)",0.0000,1.0,0.313800
4,5. CLAHE (final),"(352, 256)",0.0012,1.0,0.312100


In [23]:
from src.preprocessing.visualize import visualize_preprocessing_stages

visualize_preprocessing_stages(
    {
        "1. raw slice": raw_slice,
        f"2. resample {config.target_spacing_mm}mm + crop": geometry,
        "3. normalised [0,1]": normalised,
        f"4. {config.denoise_method} denoise": denoised,
        "5. CLAHE (final)": final,
    },
    title=f"Preprocessing stages - {row['image_id']} sagittal slice {slice_index}",
)
plt.show()

<Figure size 1700x400 with 5 Axes>

### 7.4 Evidence for the optional steps

The table below is the measured comparison that decided the denoising and
contrast choices. `class_contrast_cnr` is the one that matters most: it is the
vertebra-vs-disc separability, i.e. exactly what the project has to achieve.

In [24]:
choices_csv = paths.REPORTS_DIR / "preprocessing_choices.csv"
if choices_csv.exists():
    scores = pd.read_csv(choices_csv)
    order = ["normalised only", "median 3x3", "gaussian 3x3", "bilateral",
             "CLAHE only", "median + CLAHE", "gaussian + CLAHE"]
    means = scores.groupby("variant").mean(numeric_only=True).reindex(order)
    display(means.round(5))
    print("Best vertebra-vs-disc contrast:", means["class_contrast_cnr"].idxmax())
    print("\nNote: gaussian removes the most noise but loses the most edge sharpness,")
    print("and scores lower on class contrast than median + CLAHE.")
else:
    print("Run: python scripts/02b_justify_steps.py")

,noise_sigma,boundary_sharpness,class_contrast_cnr,dynamic_range_std,bias_inhomogeneity
variant,,,,,
normalised only,0.01048,0.11741,2.13101,0.26923,0.18995
median 3x3,0.00676,0.09430,2.29612,0.26640,0.19099
gaussian 3x3,0.00181,0.08696,2.30517,0.26176,0.18994
bilateral,0.00626,0.10422,2.26637,0.26711,0.19014
CLAHE only,0.03147,0.15839,2.25459,0.26939,0.08855
median + CLAHE,0.01232,0.11737,2.55622,0.25764,0.10964
gaussian + CLAHE,0.00613,0.12270,2.47932,0.25993,0.08445


Best vertebra-vs-disc contrast: median + CLAHE

Note: gaussian removes the most noise but loses the most edge sharpness,
and scores lower on class contrast than median + CLAHE.


---
## 8. Mask preprocessing

Masks go through the **same geometric transform** as their image, differing only
in interpolation. That is what guarantees the two stay pixel-aligned.

**Nearest neighbour is used for masks, always.** Label values are categorical:
averaging vertebra `3` and vertebra `4` gives `3.5`, which is not a structure,
and averaging a disc (`201`) with background (`0`) gives ~`100`, which happens to
be the spinal canal's label. The demonstration below counts exactly how many
non-existent label values each interpolation method invents.

In [25]:
import cv2

from src.preprocessing.transforms import center_crop_or_pad, preprocess_mask

mask_slice = mask_volume.sagittal_slice(slice_index)
original_labels = set(int(v) for v in np.unique(mask_slice))

rows_n, cols_n = mask_slice.shape
new_rows = int(round(rows_n * spacing[0] / config.target_spacing_mm))
new_cols = int(round(cols_n * spacing[1] / config.target_spacing_mm))

comparison = []
for name, flag in [("NEAREST (used)", cv2.INTER_NEAREST),
                   ("BILINEAR (unsafe)", cv2.INTER_LINEAR),
                   ("BICUBIC (unsafe)", cv2.INTER_CUBIC)]:
    resized = cv2.resize(mask_slice.astype(np.float32), (new_cols, new_rows), interpolation=flag)
    cropped = center_crop_or_pad(resized, config.target_size, pad_value=0)
    produced = set(int(round(v)) for v in np.unique(cropped))
    comparison.append({
        "method": name,
        "label values produced": len(produced),
        "INVENTED label values": len(produced - original_labels),
    })

print(f"Original mask has {len(original_labels)} label values: {sorted(original_labels)}")
pd.DataFrame(comparison)

Original mask has 15 label values: [0, 1, 2, 3, 4, 5, 6, 7, 100, 202, 203, 204, 205, 206, 207]


,method,label values produced,INVENTED label values
0,NEAREST (used),15,0
1,BILINEAR (unsafe),66,51
2,BICUBIC (unsafe),164,149


In [26]:
from src.preprocessing.visualize import visualize_interpolation_comparison

visualize_interpolation_comparison(mask_slice, spacing, config)
plt.show()

<Figure size 1700x460 with 4 Axes>

### 8.1 Mask output and integrity verification

The geometric transform is applied to the **raw** label values first, and the
semantic/instance remap happens afterwards - so resizing never operates on a
collapsed label space and each vertebra/disc identity survives the resize.

In [27]:
semantic_mask, instance_mask = preprocess_mask(mask_slice, spacing, config)

print(f"raw mask      : shape={mask_slice.shape}, labels={sorted(int(v) for v in np.unique(mask_slice))}")
print(f"semantic mask : shape={semantic_mask.shape}, dtype={semantic_mask.dtype}, "
      f"labels={np.unique(semantic_mask).tolist()}")
print(f"instance mask : shape={instance_mask.shape}, dtype={instance_mask.dtype}, "
      f"labels={np.unique(instance_mask).tolist()}")
print(f"\nimage and mask shapes identical: {final.shape == semantic_mask.shape}")

# No label may be invented by the resize; disappearance of a tiny structure is allowed.
from src.preprocessing.transforms import validate_mask_labels

check = validate_mask_labels(mask_slice, instance_mask, mapping=to_instance)
pd.Series(check, name="value").to_frame()

raw mask      : shape=(578, 448), labels=[0, 1, 2, 3, 4, 5, 6, 7, 100, 202, 203, 204, 205, 206, 207]
semantic mask : shape=(352, 256), dtype=uint8, labels=[0, 1, 2, 3]
instance mask : shape=(352, 256), dtype=uint8, labels=[0, 1, 2, 3, 4, 5, 6, 7, 10, 12, 13, 14, 15, 16, 17]

image and mask shapes identical: True


,value
labels_before,"[0, 1, 2, 3, 4, 5, 6, 7, 10, 12, 13, 14, 15, 1..."
labels_after,"[0, 1, 2, 3, 4, 5, 6, 7, 10, 12, 13, 14, 15, 1..."
labels_lost,[]
labels_invented,[]
ok,True


---
## 9. Before/after visualisation

The five required panels for randomly chosen samples: original MRI, original
mask, preprocessed MRI, preprocessed mask, and the preprocessed MRI with the
mask overlaid. Figures are also written to `outputs/visualizations/` by
`scripts/03_preprocess.py`.

Slices are displayed in standard radiological sagittal orientation: **superior
at the top, anterior on the left.**

In [28]:
from src.preprocessing.transforms import preprocess_image
from src.preprocessing.visualize import visualize_before_after, visualize_sample

rng = np.random.default_rng(paths.RANDOM_SEED)

# One sample per modality so the figures are not all the same sequence type.
selection = []
for modality in ["t1", "t2", "t2_SPACE"]:
    subset = pairs[pairs["modality"] == modality]
    selection.append(subset.iloc[int(rng.integers(len(subset)))])

for row in selection:
    image_v = load_image(paths.PROJECT_ROOT / row["image_path"])
    mask_v = load_mask(paths.PROJECT_ROOT / row["mask_path"])
    index = int(np.argmax((mask_v.array > 0).sum(axis=(0, 1))))
    sp = image_v.in_plane_spacing
    stats = intensity_statistics(image_v.array, percentiles=config.clip_percentiles)

    original_image = image_v.sagittal_slice(index)
    original_mask = mask_v.sagittal_slice(index)
    processed_image = preprocess_image(original_image, sp, config, volume_stats=stats)
    processed_mask, _ = preprocess_mask(original_mask, sp, config)

    visualize_sample(
        original_image, original_mask, processed_image, processed_mask,
        title=(f"{row['image_id']} | patient {row['patient_id']} | {row['modality']} | "
               f"slice {index} | in-plane {sp[0]:.3f} x {sp[1]:.3f} mm"),
    )
    plt.show()

<Figure size 1800x440 with 5 Axes>

<Figure size 1800x440 with 5 Axes>

<Figure size 1800x440 with 5 Axes>

In [29]:
# Intensity-focused before/after: the histograms show what normalisation achieved.
row = selection[0]
image_v = load_image(paths.PROJECT_ROOT / row["image_path"])
mask_v = load_mask(paths.PROJECT_ROOT / row["mask_path"])
index = int(np.argmax((mask_v.array > 0).sum(axis=(0, 1))))
stats = intensity_statistics(image_v.array, percentiles=config.clip_percentiles)

original_image = image_v.sagittal_slice(index)
processed_image = preprocess_image(original_image, image_v.in_plane_spacing, config,
                                   volume_stats=stats)

visualize_before_after(
    original_image, processed_image,
    title=(f"Before vs after - {row['image_id']} "
           f"(raw range [{original_image.min():.0f}, {original_image.max():.0f}])"),
)
plt.show()

<Figure size 1100x800 with 4 Axes>

### 9.1 Consistency across the dataset

The point of preprocessing is that every sample now looks structurally the same
to a model: identical size, identical orientation, identical intensity scale.

In [30]:
from src.preprocessing.visualize import visualize_dataset_grid

samples = []
for _ in range(6):
    row = pairs.iloc[int(rng.integers(len(pairs)))]
    image_v = load_image(paths.PROJECT_ROOT / row["image_path"])
    mask_v = load_mask(paths.PROJECT_ROOT / row["mask_path"])
    index = int(np.argmax((mask_v.array > 0).sum(axis=(0, 1))))
    sp = image_v.in_plane_spacing
    stats = intensity_statistics(image_v.array, percentiles=config.clip_percentiles)
    processed_mask, _ = preprocess_mask(mask_v.sagittal_slice(index), sp, config)
    samples.append({
        "image": preprocess_image(image_v.sagittal_slice(index), sp, config, volume_stats=stats),
        "mask": processed_mask,
        "label": f"{row['image_id']}\n{row['modality']}",
    })

visualize_dataset_grid(
    samples,
    title=(f"Preprocessed samples - all {config.target_size[0]}x{config.target_size[1]} px "
           f"at {config.target_spacing_mm} mm/px"),
)
plt.show()

<Figure size 1560x600 with 12 Axes>

---
## 10. Train / validation / test split

Run the full pipeline first if `data/processed/` is empty:

```bash
python scripts/03_preprocess.py     # writes every preprocessed slice
python scripts/04_split.py          # writes the patient-level split
```

**The split is made on `patient_id`, never on individual slices.** Three
independent reasons, all established by the inspection above:

1. A series contributes 8-154 adjacent sagittal slices that are nearly
   identical to their neighbours.
2. A patient contributes 1-3 series of the *same* anatomy.
3. The T1 and T2 masks of a patient are frequently byte-identical.

Random slice splitting would put near-copies of the same image in both training
and test, and the resulting Dice/IoU would be optimistically biased.

In [31]:
from src.preprocessing.splits import (
    create_dataset_split, split_fraction_table, summarise_split, verify_no_leakage,
)

slice_index = pd.read_csv(paths.SLICE_INDEX_CSV) if paths.SLICE_INDEX_CSV.exists() else None
if slice_index is None:
    print("data/processed/slice_index.csv not found - run scripts/03_preprocess.py first.")

split_frame = create_dataset_split(
    pairs, fractions={"train": 0.70, "val": 0.15, "test": 0.15},
    seed=paths.RANDOM_SEED, stratify=True,
)

if slice_index is not None:
    slice_index["split"] = slice_index["image_id"].map(
        split_frame.set_index("image_id")["split"]
    )

leakage = verify_no_leakage(split_frame)
print(f"No patient in more than one split : {leakage['ok']}")
print(f"Overlapping patients              : {leakage['overlaps'] or 'none'}")
print(f"Sum of per-split patient counts   : {leakage['total_ids_counted']}")
print(f"Distinct patients in the dataset  : {leakage['total_ids_unique']}")
print("  (the last two matching proves the split is a true partition)")

No patient in more than one split : True
Overlapping patients              : none
Sum of per-split patient counts   : 218
Distinct patients in the dataset  : 218
  (the last two matching proves the split is a true partition)


In [32]:
counts = summarise_split(split_frame, slice_index)
table = split_fraction_table(counts)
display(table)

if slice_index is not None:
    from src.preprocessing.visualize import visualize_split
    visualize_split(counts)
    plt.show()

,split,patients,patients_pct,series,series_pct,slices,slices_pct
0,train,152,69.7,319,71.4,9128,73.5
1,val,33,15.1,64,14.3,1632,13.1
2,test,33,15.1,64,14.3,1655,13.3


<Figure size 750x440 with 1 Axes>

In [33]:
# Composition per split: stratification keeps them comparable.
composition = []
for name in ["train", "val", "test"]:
    rows_ = split_frame[split_frame["split"] == name]
    composition.append({
        "split": name,
        "patients": rows_["patient_id"].nunique(),
        "series": len(rows_),
        "% female": round(100 * (rows_["sex"] == "F").mean(), 1),
        "mean vertebrae": round(rows_["num_vertebrae"].mean(), 2),
        "mean discs": round(rows_["num_discs"].mean(), 2),
        "t1 / t2 / SPACE": "{} / {} / {}".format(
            int((rows_["modality"] == "t1").sum()),
            int((rows_["modality"] == "t2").sum()),
            int((rows_["modality"] == "t2_SPACE").sum()),
        ),
    })
pd.DataFrame(composition)

,split,patients,series,% female,mean vertebrae,mean discs,t1 / t2 / SPACE
0,train,152,319,61.1,7.01,7.07,140 / 147 / 32
1,val,33,64,62.5,6.91,6.95,28 / 32 / 4
2,test,33,64,64.1,6.97,6.98,28 / 31 / 5


Note that `overview.csv` ships its own `subset` column, but it only separates
training from validation (no test set) and is defined per *series*, so Sprint 1
derives its own patient-level three-way split.

In [34]:
if "subset" in split_frame.columns:
    display(pd.crosstab(split_frame["subset"], split_frame["split"],
                        margins=True, margins_name="total"))

split,test,train,val,total
subset,,,,
training,47,260,53,360
validation,17,59,11,87
total,64,319,64,447


---
## 11. Final preprocessing summary

In [35]:
report_json = paths.REPORTS_DIR / "preprocessing_report.json"
if report_json.exists():
    with open(report_json) as handle:
        preprocessing_summary = json.load(handle)

    size = preprocessing_summary["dataset_size"]
    before = preprocessing_summary["dimensions_before"]
    after = preprocessing_summary["dimensions_after"]
    integrity = preprocessing_summary["mask_integrity"]

    display(pd.Series({
        "Series processed": f"{size['series_succeeded']} / {size['series_attempted']}",
        "Patients": size["patients"],
        "Sagittal slices available": size["slices_available"],
        "Preprocessed slices written": size["slices_written"],
        "Slices dropped (too little annotation)": size["slices_dropped_unannotated"],
        "Distinct input shapes": before["distinct_shapes"],
        "Output shape": f"{after['rows']} x {after['cols']} px @ {after['spacing_mm']} mm",
        "Series where a label was invented": integrity["series_with_invented_labels"],
        "Annotated area retained (mean)": f"{100 * integrity['annotated_area_retained_mean']:.2f}%",
        "Annotated area retained (worst)": f"{100 * integrity['annotated_area_retained_min']:.2f}%",
    }, name="value").to_frame())

    print("\nSemantic class distribution (% of all pixels):")
    display(pd.Series(preprocessing_summary["class_distribution_pct_of_all"],
                      name="% of pixels").to_frame())
else:
    print("Run: python scripts/03_preprocess.py")

,value
Series processed,447 / 447
Patients,218
Sagittal slices available,14070
Preprocessed slices written,12415
Slices dropped (too little annotation),1655
Distinct input shapes,149
Output shape,352 x 256 px @ 1.0 mm
Series where a label was invented,0
Annotated area retained (mean),100.06%
Annotated area retained (worst),99.33%



Semantic class distribution (% of all pixels):


,% of pixels
background,94.7705
vertebra,3.7570
intervertebral_disc,0.7901
spinal_canal,0.6824


In [36]:
# Load a preprocessed slice back from disk to confirm the saved artefacts are usable.
from src.preprocessing.dataset import load_processed_slice

if slice_index is not None and len(slice_index):
    sample_row = slice_index.iloc[0]
    bundle = load_processed_slice(sample_row["npz_path"])
    print(f"slice_id : {sample_row['slice_id']}")
    print(f"file     : {sample_row['npz_path']}")
    for key, array in bundle.items():
        print(f"  {key:14s} shape={array.shape} dtype={array.dtype} "
              f"range=[{array.min()}, {array.max()}]")

    from src.preprocessing.visualize import overlay_mask
    fig, axes = plt.subplots(1, 3, figsize=(11, 4.4))
    axes[0].imshow(bundle["image"], cmap="gray"); axes[0].set_title("saved image")
    axes[1].imshow(bundle["mask"], cmap="nipy_spectral", interpolation="nearest")
    axes[1].set_title("saved semantic mask")
    axes[2].imshow(overlay_mask(bundle["image"], bundle["mask"])); axes[2].set_title("overlay")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(f"Round-trip check: {sample_row['slice_id']}")
    fig.tight_layout()
    plt.show()

slice_id : 1_t1_s012
file     : data/processed/slices/1_t1_s012.npz
  image          shape=(352, 256) dtype=float32 range=[0.0011138916015625, 1.0]
  mask           shape=(352, 256) dtype=uint8 range=[0, 2]
  mask_instance  shape=(352, 256) dtype=uint8 range=[0, 15]


<Figure size 1100x440 with 3 Axes>

### What Sprint 1 delivered

* Clean project structure; the two EHR tutorial notebooks preserved unmodified in
  `tutorials/` as reference material only.
* Raw dataset left byte-for-byte intact in `data/raw/`; extraction is scripted
  and reproducible.
* All 447 volume pairs inspected: format, geometry, label vocabulary, intensity
  conventions, duplicates and anatomical consistency.
* Image-mask pairing driven by parsed filename identifiers, with every pair
  verified.
* A preprocessing pipeline whose every optional step is backed by a measurement.
* Masks resized with nearest neighbour only, with automated verification that no
  label value was invented.
* Reproducible, stratified, **patient-level** train/validation/test split with an
  explicit leakage check.
* Reports in `outputs/preprocessing_reports/` and figures in
  `outputs/visualizations/`.

### Dataset problems found

* Pixel spacing, matrix size, slice thickness and stored orientation all vary
  between series - handled by RAS reorientation and resampling to a common
  physical scale.
* Two incompatible intensity conventions - handled by foreground-restricted
  percentile normalisation.
* 107 groups of byte-identical mask volumes, all within one patient (shared
  T1/T2 annotation) - makes patient-level splitting mandatory.
* `sex` in `overview.csv` has trailing-whitespace duplicates - stripped on load.
* A small number of series annotate an unusually short span of the spine - kept
  but flagged.
* Severe foreground/background class imbalance, with intervertebral discs the
  smallest target class.

### Next sprint

1. Baseline segmentation model (U-Net) on the preprocessed 2-D slices.
2. Class-imbalance-aware loss (Dice or compound Dice + cross-entropy).
3. Data augmentation.
4. Post-processing (largest-component filtering, morphological cleanup).
5. Evaluation on the held-out **test patients**: Dice, IoU, precision, recall.
6. Qualitative comparison of predictions against ground truth.

> Sprint 1 reports no accuracy metric. Dice and IoU will be reported only after a
> model has actually been trained and evaluated.